In [6]:
# Basic imports

import os
import pandas as pd
from dotenv import load_dotenv
from langchain.tools import tool
from langchain.agents import create_agent 
from langchain_google_genai import ChatGoogleGenerativeAI

#load environment variables
load_dotenv()

True

In [7]:
#load the csv file into a pandas dataframe

transactions = pd.read_csv('datasets/transactions.csv')
budget = pd.read_csv('datasets/budget.csv')
bills = pd.read_csv('datasets/bills.csv')
income = pd.read_csv('datasets/income.csv')

In [8]:
@tool
def transactions_analyzer(query: str) -> str:
    """Analyze transactions, categorize spending, and summarize expenses.

    Args:
        query : the query to check category (e.g."Food & Drinks","Entertainment") 

    Returns:
        Total amount spend on each category 
    """
    try:
        transactions['Amount'] = pd.to_numeric(transactions['Amount'], errors='coerce')

        # category mapping
        category_map = {
            "Food":"Food & Drinks",
            "Drink":"Food & Drinks",
            "Entertainment":"Entertainment",
            "Utilities":"Utilities",
            "Groceries":"Groceries",
            "Transport":"Transport"    
        }

        # look for matching category in the query
        matched_category = None
        for keyword , category_name in category_map.items():
            if keyword.lower() in query.lower():
                matched_category = category_name
                break
        if matched_category:
            total = transactions[transactions["Category"] == matched_category]['Amount'].sum()
            return f"Total for {matched_category} is: ${total:.2f}"
        return "Query not understood"
    except Exception as e:
        return f"Error accessing transactions{str(e)}"


In [ ]:
import datetime


@tool
def bill_reminder(query:str) -> str:
    """"Check upcoming or unpaid bills and calculate totals

    Args:
        query : the query to check bills (e.g."paid bills","unpaid bills")
    Returns:
        Paid or unpaid bills with total amount ,bill names ,and due dates
    """
    try:
        bills['Amount'] = pd.to_numeric(bills['Amount'],errors='coerce')

        if "paid" in query.lower():
            paid_bills = bills[bills['Status'] == 'Paid']['Amount'].sum()
            bill_names = bills[bills['Status'] == 'Paid']['Bill Name'].tolist()
            return f"Total paid bills: ${paid_bills:.2f}. Paid bills: {', '.join(bill_names)}"
        
        elif "unpaid" in query.lower():
            unpaid_bills = bills[bills['Status'] == 'Unpaid']['Amount'].sum()
            bill_names = bills[bills['Status'] == 'Unpaid']['Bill Name'].tolist()
            return f"Total unpaid bills:${unpaid_bills:.2f}. Unpaid bills: {','.join(bill_names)}"
        
        elif "week" in query.lower():
            today = datetime.datetime.now()
            week_end = today + datetime.timedelta(days=7)
            result = bills[(bills['Due Date'] >= today) & (bills['Due Date'] <= week_end)]
            return f"Upcoming bills in the next week: {result.to_string(index=False)}"
        
        elif "month" in query.lower():
            today = datetime.datetime.now()
            result = bills[bills['Due Date'].dt.month == today.month]
            return f"Upcoming bills in the current month: {result.to_string(index=False)}"
        else:
            return "Query not understood..Specify 'paid bills', 'unpaid bills', 'bills due this week' or 'bills due this month'"
    except Exception as e:
        return f"Error accessing bills: {str(e)}"


In [ ]:
@tool
def budget_tracker(query:str) -> str:
    """Check the budget for each category

    Args : the query to check category (e.g."Food & Drinks","Entertainment")

    Returns:
        Category names and total amount of the each budget category 
    """

    try:
        budget['Budget Amount'] = pd.to_numeric(budget['Budget Amount'],errors='coerce')

        category_map = {
            "Food":"Food & Drinks",
            "Drink":"Food & Drinks",
            "Entertainment":"Entertainment",
            "Utilities":"Utilities",
            "Groceries":"Groceries",
            "Transport":"Transport" 
        }

        matched_category = None
        for keyword,category_name in category_map.items():
            if keyword.lower() in query.lower():
                matched_category = category_name
                break
        if matched_category:
            budgeted_amount = budget[budget['Category'] == matched_category]['Budget Amount'].sum()
            actual_spending = transactions[transactions['Category'] == matched_category]['Amount'].sum()
            return f"Budget for {matched_category} is: ${budgeted_amount: .2f}, Actual spending is: ${actual_spending: .2f}."
        return "Query not understood"

    except Exception as e:
        return f"Error accessing budget: {str(e)}"

In [ ]:
@tool
def income_tracker(query:str) -> str:
    """"""

SyntaxError: incomplete input (1492007124.py, line 3)

In [15]:
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    google_api_key = os.getenv("GEMINI-API-KEY"),
    max_tokens = 1024,
    temperature = 0
    )

In [11]:
tools =[transactions_analyzer,bill_reminder]

In [12]:
# create the agent
agent = create_agent(
    model=llm,
    tools=tools,
)

In [ ]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "How much did I spend on Entertainment last month?"}
    ]
})

print(response["messages"][-1].content)


{'messages': [HumanMessage(content='How much did I spend on Entertainment last month?', additional_kwargs={}, response_metadata={}, id='02959b42-fba3-4fda-b13a-05e149507a50'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'transactions_analyzer', 'arguments': '{"query": "Entertainment"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c671e-10ad-7801-8183-221d4858ffa8-0', tool_calls=[{'name': 'transactions_analyzer', 'args': {'query': 'Entertainment'}, 'id': '371452f0-ab8c-4189-a790-8ea5bc105094', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 93, 'output_tokens': 15, 'total_tokens': 108, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='Total for Entertainment is: $6312.38', name='transactions_analyzer', id='344b76b0-a7a9-4fcc-824e-67ccde1f3312', tool_call_id='371452f0-ab8c-4189-a790-8ea5bc105094'), AIMessage(c